In [1]:
import os
import glob

import json
import rasterio
from rasterio.mask import mask
#from patchify import patchify
import numpy as np 
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon,box

In [2]:
FILEPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training'

EXCELPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training\Phase_1_Dataset_Details.xlsx'

DESTINATION_PRE_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE'
DESTINATION_POST_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\POST'

In [4]:

'''
'Call_970_Afghanistan',

'''

COUNTRIES_NEED = ['Call_1075_China']

class PathCollector:

    def __init__(self, file_path, excel_path):
        self.file_path = file_path
        self.excel_path = excel_path
        self.gdf = gpd.GeoDataFrame(columns=["pre", "post"])  

    def collect(self):
        pre, post = [], []

        if os.path.isfile(self.excel_path):
            df = pd.read_excel(self.excel_path)
            pre_list = df.loc[df["Pre or Post"]=="Pre-event"]["Dataset ID"].to_list()
            post_list = df.loc[df["Pre or Post"]=="Post-event"]["Dataset ID"].to_list()
        else:
            raise FileNotFoundError("Excel file not found!")

        if os.path.isdir(self.file_path):
            countries = os.listdir(self.file_path)
            for sing_count in countries:
                if sing_count in COUNTRIES_NEED:
                    country_path = os.path.join(self.file_path, sing_count)
                    country_level = os.listdir(country_path)

                    for items in pre_list:
                        if items in country_level:
                            pre_path = os.path.join(country_path, items, "Optical_Calibration", "overview-trc_PROJ.tif")
                            pre.append(pre_path)

                    for items in post_list:
                        if items in country_level:
                            post_path = os.path.join(country_path, items, "Optical_Calibration", "overview-trc_PROJ.tif")
                            post.append(post_path)


        self.gdf = gpd.GeoDataFrame(
            [{"pre": p, "post": d} for p, d in zip(pre, post)]
        )

        return self.gdf

In [ ]:
class GEO_PreProssessing:

    def __init__(self, FILEPATH,EXCELPATH,DESTINATION_PRE_PATH,DESTINATION_POST_PATH,chip_size = (224,224,3)):
        
        self.filepath = FILEPATH
        self.excelpath = EXCELPATH

        self.pre_destination = DESTINATION_PRE_PATH
        self.post_destination = DESTINATION_POST_PATH

        self.chip_size = chip_size

        collector = PathCollector(self.filepath,self.excelpath)
        self.gdf = collector.collect()

    def open(self,path,no_channels=[1,2,3],transpose=True):
        '''
        Read the image 
        Return: Image in numpy array format as (H,W,C) and Source format
        '''
        src = rasterio.open(path)
        img = src.read(no_channels)
        if transpose:
              img.transpose(1,2,0)
              return img,src
        else:
              return img,src

    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels):
        
        '''
        Find the overlapping area an save both raster within intersection extent
        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        
        '''

        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })
    

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:
            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def makeChips(self,in_pre,in_post,out_pre,out_post,no_channels):
        pass    
          
        ''' 
    
        pre_clip = pre_clip.transpose(1, 2, 0)
        post_clip = post_clip.transpose(1, 2, 0)

        pre_patches = patchify(pre_clip,self.chip_size,step=224)
        pre_patches = np.squeeze(pre_patches).reshape(-1,224,224,3)

        post_patches = patchify(post_clip,self.chip_size,step=224)
        post_patches = np.squeeze(post_patches).reshape(-1,224,224,3)

        for i,images in enumerate(zip(pre_patches,post_patches),start=1):

            pre,post = images

            pre = pre.transpose(2,0,1)
            post = post.transpose(2,0,1)

            pre_meta.update({
            "height": pre.shape[1],
            "width": pre.shape[2],
            "count": pre.shape[0],
            "transform": pre_transform
            })

            post_meta.update({
            "height": post.shape[1],
            "width": post.shape[2],
            "count": post.shape[0],
            "transform": post_transform
            })

            if os.path.exists(self.pre_destination):
                    with rasterio.open(os.path.join(self.pre_destination,f'./pre_image{i}.tiff'),'w',**pre_meta) as pre_dest:
                        pre_dest.write(pre)
            else:
                print('Defined folder does not exists: Saving PRE to the current directory')
                with rasterio.open(f'./pre_image{i}.tiff','w',**pre_meta) as pre_dest:
                    pre_dest.write(pre)

            if os.path.exists(self.post_destination):
                with rasterio.open(os.path.join(self.post_destination,f'./post_image{i}.tiff'),'w',**post_meta) as post_dest:
                    post_dest.write(post)
            else:
                print('Defined folder does not exists: Saving POST to the current directory')
                with rasterio.open(f'./post_image{i}.tiff','w',**post_meta) as post_dest:
                    post_dest.write(post)

'''

    def Mean_Std(self,IMAGE_FOLDER_PATH,no_channels,transpose):
        '''
        Computing the Mean and Std for the whole dataset

        Args:
            IMAGE_FOLDER_PATH(str): Provide the directory consist of images (SAMPLE: ./ROOT_DATA_FOLDER/PRE_FOLDER)
        '''
        
        try:

            for path in os.listdir(IMAGE_FOLDER_PATH):
                img,src= open(path,no_channels,transpose)
                pixels = img.reshape(img.shape[0], -1).T 

                if mean is None:
                    mean = np.zeros(pixels.shape[1])
                    M2 = np.zeros(pixels.shape[1])

                n = pixels.shape[0]
                new_mean = np.mean(pixels, axis=0)
                new_M2 = np.var(pixels, axis=0) * n

                delta = new_mean - mean
                total_n = n_pixels + n
                mean += delta * n / total_n
                M2 += new_M2 + delta**2 * n_pixels * n / total_n
                n_pixels = total_n

            std = np.sqrt(M2 / n_pixels)
            print(f'[MEAN]----{mean}')
            print(f'[STD]----{std}')
        
        except ValueError:
            print('PATH is invalid !')

In [16]:
process = GEO_PreProssessing(FILEPATH,EXCELPATH,DESTINATION_PRE_PATH,DESTINATION_POST_PATH)

In [17]:
img,src= process.open(r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE\pre3.tif')
img

array([[[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       ...,

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535]],

       [[65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        ...,
        [65535, 65535, 65535],
        [65535, 65535, 65535],
        [65535,

In [18]:
img.shape

(32208, 21858, 3)

In [178]:
def open(path):
        '''
        Read the image
        '''
        with rasterio.open(path) as src:
            img = src.read([1,2,3])

        return img,src


img,_ = open(r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE\pre_image1.tiff')

In [179]:
img.shape

(3, 6932, 6406)